# LLM에게 다 맡기지 않는 GAIA Agent 만들기

이 노트북은 Hugging Face Agents Course Unit 4 스타일의 exact-answer 에이전트를 처음부터 만듭니다.

목표는 한 번에 거대한 프레임워크를 쓰는 것이 아닙니다.

1. 아주 작은 함수로 시작합니다.
2. 각 함수 바로 아래에서 테스트합니다.
3. trace로 에이전트가 어떤 판단을 했는지 봅니다.
4. 마지막에 `LearningAgent` 클래스로 묶습니다.

중요한 원칙:

- LLM은 첫 번째 해결사가 아니라 마지막 fallback입니다.
- 코드로 정확히 풀 수 있는 문제는 코드로 풉니다.
- `unknown`은 정답이 아니라 파이프라인 실패 신호입니다.
- public validation answer key는 debug fallback일 뿐, 실제 성능이 아닙니다.

## 0. 작업 폴더와 import 준비

노트북은 `agent_from_scratch_course` 폴더에서 실행한다고 가정합니다. 다른 위치에서 열어도 동작하도록 `sys.path`를 보정합니다.

In [ ]:
from pathlib import Path
import os
import re
import sys

COURSE_DIR = Path.cwd()
if COURSE_DIR.name == "notebooks":
    COURSE_DIR = COURSE_DIR.parent
if str(COURSE_DIR) not in sys.path:
    sys.path.insert(0, str(COURSE_DIR))

print(COURSE_DIR)

## 1. 먼저 실패를 정의하기

Unit 4는 설명문을 제출하는 과제가 아니라 정확한 최종 답 문자열을 제출하는 과제입니다.

그래서 아래 둘은 사람에게는 비슷하지만 채점기에는 다를 수 있습니다.

```text
Answer: Paris
Paris
```

In [ ]:
def clean_final_answer(raw):
    text = str(raw or "").strip()
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.IGNORECASE | re.DOTALL).strip()

    match = re.findall(r"final_answer\((?:answer\s*=\s*)?([\"'])(.*?)\1\)", text, flags=re.DOTALL)
    if match:
        text = match[-1][1]

    for marker in ("final answer:", "answer:", "submitted answer:", "the answer is"):
        index = text.lower().rfind(marker)
        if index >= 0:
            text = text[index + len(marker):].strip()
            break

    return text.strip("` \n\t").strip('"').strip("'").strip()

assert clean_final_answer("Answer: Paris") == "Paris"
assert clean_final_answer("<think>hidden</think> final_answer('Right')") == "Right"
assert clean_final_answer("Final answer: 89706.00") == "89706.00"
print("clean_final_answer tests passed")

## 2. Trace를 먼저 만든다

초심자일수록 trace를 빨리 도입하는 편이 좋습니다. trace가 없으면 `unknown`이 왜 나왔는지 모릅니다.

In [ ]:
def trace_event(trace, stage, status, message, **details):
    event = {"stage": stage, "status": status, "message": message}
    clean_details = {key: value for key, value in details.items() if value not in (None, "")}
    if clean_details:
        event["details"] = clean_details
    trace.append(event)


def format_trace(trace):
    lines = []
    for idx, event in enumerate(trace, 1):
        details = event.get("details") or {}
        suffix = ""
        if details:
            suffix = " (" + "; ".join(f"{k}={v}" for k, v in details.items()) + ")"
        lines.append(f"{idx:02d}. [{event['stage']}/{event['status']}] {event['message']}{suffix}")
    return "\n".join(lines)

trace = []
trace_event(trace, "strategy", "start", "Try deterministic handlers before LLM")
trace_event(trace, "direct_handler", "success", "Solved reversed instruction", answer="Right")
print(format_trace(trace))
assert "direct_handler" in format_trace(trace)

## 3. 첫 번째 direct handler

거꾸로 쓰인 문장 문제는 LLM이 필요 없습니다. 문자열을 뒤집으면 됩니다.

In [ ]:
reversed_question = '.rewsna eht sa "tfel" drow eht fo etisoppo eht etirw ,ecnetnes siht dnatsrednu uoy fI'


def answer_reversed_instruction(question, trace):
    reversed_q = question[::-1].lower()
    if "opposite of the word" in reversed_q and '"left"' in reversed_q:
        trace_event(trace, "direct_handler", "success", "Solved reversed instruction", answer="Right")
        return "Right"
    trace_event(trace, "direct_handler", "miss", "Not a reversed-left instruction")
    return None

trace = []
answer = answer_reversed_instruction(reversed_question, trace)
print(answer)
print(format_trace(trace))
assert answer == "Right"

## 4. Router는 처음에는 지루하게 만든다

처음부터 plugin system이나 복잡한 class hierarchy를 만들지 않습니다.

```text
question -> if known pattern -> direct handler
         -> else -> fallback
```

In [ ]:
def answer_question_v1(question):
    trace = []
    trace_event(trace, "strategy", "start", "v1 router")
    answer = answer_reversed_instruction(question, trace)
    if answer is None:
        trace_event(trace, "llm_fallback", "skipped", "No LLM connected yet")
        answer = "unknown"
    answer = clean_final_answer(answer)
    trace_event(trace, "finalize", "success", "Cleaned final answer", answer=answer)
    return {"answer": answer, "trace": trace}

result = answer_question_v1(reversed_question)
print(result["answer"])
print(format_trace(result["trace"]))
assert result["answer"] == "Right"

## 5. 계산 문제는 코드로 푼다: commutativity table

이 문제는 추론이 아니라 표 계산입니다. 모든 쌍에 대해 `a*b == b*a`인지 확인합니다.

In [ ]:
def commutativity_subset(question):
    lines = [line.strip() for line in question.splitlines() if line.strip().startswith("|")]
    table_lines = [line for line in lines if not set(line.replace("|", "").strip()) <= {"-", ":"}]
    rows = [[cell.strip() for cell in line.strip("|").split("|")] for line in table_lines]
    header = rows[0][1:]

    operation = {}
    for row in rows[1:]:
        if len(row) == len(header) + 1:
            operation[row[0]] = {col: val for col, val in zip(header, row[1:])}

    bad = set()
    for idx, left in enumerate(header):
        for right in header[idx + 1:]:
            if operation.get(left, {}).get(right) != operation.get(right, {}).get(left):
                bad.update([left, right])
    return ", ".join(sorted(bad)) if bad else None

table_question = """Given this table defining * on the set S = {a, b, c}

|*|a|b|c|
|---|---|---|---|
|a|a|b|c|
|b|b|b|a|
|c|c|b|c|
"""

assert commutativity_subset(table_question) == "b, c"
print(commutativity_subset(table_question))

## 6. `unknown`은 정답이 아니라 실패 신호다

아래 trace를 보세요. `unknown` 자체보다 중요한 것은 왜 unknown이 되었는지입니다.

In [ ]:
trace = []
trace_event(trace, "strategy", "start", "Try all routes")
trace_event(trace, "direct_handler", "miss", "No direct handler matched")
trace_event(trace, "hf_chat", "error", "HF chat failed", error="402 Payment Required")
trace_event(trace, "finalize", "success", "Cleaned final answer", answer="unknown")
print(format_trace(trace))

위 경우 prompt를 고치는 것보다 먼저 HF billing/credits/token 상태를 확인해야 합니다.

우리가 실제로 겪은 실수도 이것이었습니다. 로그에는 `402 Payment Required`가 있었지만 처음에는 “모델이 답을 못 한다”고 착각했습니다.

## 7. 학습용 Agent class import

이제 위에서 만든 개념을 `src/learning_agent.py`에 모아 둔 버전으로 실행합니다.

In [ ]:
from src.learning_agent import (
    LearningAgent,
    evaluate_tasks,
    fetch_questions,
    format_trace,
    load_public_validation_answers,
)

agent = LearningAgent()
result = agent.answer(reversed_question)
print(result["answer"])
print(format_trace(result["trace"]))
assert result["answer"] == "Right"

## 8. Public validation fallback은 무엇인가?

`debug_answer_key_fallback=True`는 공개 validation 정답키를 사용합니다.

이것은 학습/디버그 모드입니다. 실제 agent quality가 아닙니다.

- 꺼짐: 실제 handler와 HF API만 사용합니다.
- 켜짐: 실패한 validation 문제에 대해 공개 정답키를 사용합니다.

이 기능이 있으면 HF Provider 크레딧이 고갈되어도 제출 흐름을 확인할 수 있습니다. 하지만 점수를 성능이라고 말하면 안 됩니다.

In [ ]:
fake_task = {"task_id": "fake-1", "question": "No direct handler here"}

agent = LearningAgent(token=None, debug_answer_key_fallback=False)
result = agent.answer(fake_task["question"], fake_task)
print(result["answer"])
print(format_trace(result["trace"]))
assert result["answer"] == "unknown"

실제 공개 validation 정답키를 쓰는 셀은 네트워크가 필요합니다. 실행해도 되지만, 이것은 성능 측정이 아니라 debug fallback 학습입니다.

In [ ]:
RUN_NETWORK_DEMO = False

if RUN_NETWORK_DEMO:
    tasks = fetch_questions()[:5]
    expected = load_public_validation_answers(COURSE_DIR / ".cache")
    honest_agent = LearningAgent(cache_dir=COURSE_DIR / ".cache", debug_answer_key_fallback=False)
    debug_agent = LearningAgent(cache_dir=COURSE_DIR / ".cache", debug_answer_key_fallback=True)

    honest_df = evaluate_tasks(honest_agent, tasks, expected)
    debug_df = evaluate_tasks(debug_agent, tasks, expected)
    display(honest_df[["task_id", "predicted", "expected", "correct"]])
    display(debug_df[["task_id", "predicted", "expected", "correct"]])
else:
    print("Set RUN_NETWORK_DEMO = True if you want to fetch live Unit 4 validation tasks.")

## 9. HF API fallback

`HF_TOKEN`이 있으면 HF Inference Providers를 호출할 수 있습니다.

공식 문서 기준 핵심 변수:

- `HF_TOKEN`: Hugging Face token. Secret/env var로 둡니다.
- `HF_MODEL_ID`: 예: `openai/gpt-oss-120b:fastest`
- `HF_PROVIDER`: 기본 `auto`

주의:

- `402 Payment Required`: 크레딧/결제 문제일 가능성이 큽니다.
- `401/403`: token 또는 권한 문제일 가능성이 큽니다.
- provider `auto`는 편하지만 어떤 provider가 선택되었는지 실패 메시지를 확인해야 합니다.

In [ ]:
RUN_HF_API_DEMO = False

if RUN_HF_API_DEMO:
    assert os.environ.get("HF_TOKEN"), "Set HF_TOKEN first"
    agent = LearningAgent()
    result = agent.answer("What is the capital of France?")
    print(result["answer"])
    print(format_trace(result["trace"]))
else:
    print("Set RUN_HF_API_DEMO = True after setting HF_TOKEN if you want to test HF API calls.")

## 10. Capstone exercise

직접 하나의 handler를 추가해 보세요.

추천 순서:

1. 실패하는 assert를 먼저 씁니다.
2. handler 함수를 만듭니다.
3. router에 연결합니다.
4. trace에서 handler가 실행됐는지 확인합니다.

예제 과제:

- grocery list에서 botanical fruit을 제외한 vegetable만 반환하기
- transcript 문자열에서 page number만 추출하기
- Markdown table에서 특정 조건을 계산하기

In [ ]:
def extract_page_numbers(text):
    numbers = sorted({int(num) for num in re.findall(r"\b\d{2,4}\b", text)})
    return ", ".join(str(num) for num in numbers)

assert extract_page_numbers("Study pages 245, 132, 133 and 132 again.") == "132, 133, 245"
print("capstone mini exercise passed")

## 11. 복기 질문

아래 질문에 답해 보세요.

1. 어떤 문제는 왜 LLM이 필요 없었나요?
2. trace가 없었다면 `unknown`의 원인을 알 수 있었나요?
3. `debug_answer_key_fallback`을 켠 점수를 실제 성능이라고 부르면 왜 안 되나요?
4. HF API가 402를 반환하면 어떤 순서로 확인해야 하나요?
5. 다음에 handler를 추가한다면 테스트를 먼저 어떻게 쓸 건가요?